# Etapa 2 — Visão Geral

Detecção de rostos reais vs. gerados por IA, com foco em **generalização cross-generator**: treinar num gerador (StyleGAN) e detectar imagens de **outros** geradores.

## O problema central
Classificar imagens do mesmo gerador de treino é trivial (ResNet-50 ~0,997 AUC), mas generalizar para outros geradores despenca (~0,63). A causa: o detector se vicia na **digital espectral** que cada gerador deixa (artefatos de upsampling) em vez de aprender o que é "sintético" em geral.

## Abordagem
1. **Diagnosticar** a digital (análise espectral + Grad-CAM + scanner multi-canal).
2. **Removê-la** com augmentation de remoção de fingerprint (BARS/MEAN, Wesselkamp et al. 2022), forçando o modelo a usar pistas que transferem entre geradores.
3. **Confirmar** o ganho cross-generator e treinar os modelos finais (ResNet-50, EfficientNet-B2).

## Bases de dados
- **140k Real and Fake Faces** (treino / in-distribution): 70k reais (CelebA) + 70k sintéticas (StyleGAN), balanceada.
- **ArtiFact (faces)** (teste cross-generator): rostos de 8 geradores diferentes (sem StyleGAN) + reais (ffhq/celebahq/metfaces). Dividido em metades `dev` (seleção) e `test` (número final) para evitar vazamento.

## Estrutura
- `notebooks_140k/` — pipeline completo.
- `notebooks_140k/aug_utils.py` — transformações e datasets em módulo importável (permite `num_workers>0` no Windows).
- `data/raw/` — 140k e ArtiFact.
- `artifacts/` — modelos, fingerprints e resultados (não versionado).
- `reports/figures/` — figuras.

## Ordem de execução
1. `01_140k_preparacao` · `01a_artifact_preparacao` — dados
2. `01b_analise_espectral` · `01c_grad_cam` · `01g_scanner_fingerprint` — diagnóstico da digital
3. `01h_remocao_fingerprint` — método BARS/MEAN: quais transformações e intensidades destroem a digital
4. `01i_treino_cross_generator` — confirma o ganho cross-generator com CNN
5. `03_treinamento_resnet50` · `04_treinamento_efficientnet_b2` — modelos finais

Os notebooks `01d/01e/01f` (busca empírica de augmentation/HP) e `02` (degradação fixa) foram a fase exploratória anterior ao método; ficam no histórico do git.